# Lorenz System Analysis (PyTorch SINDy Autoencoder)

This notebook replicates the analysis from Champion et al. (2019) for the Lorenz system using a PyTorch-based SINDy-autoencoder.

In [12]:
import torch
import pickle
import numpy as np
from autoencoder_torch import full_network

## Loading the saved model

In [ ]:
# Load params used for training
timestamp = "2025_05_14_18_26_02_575880"
params_path = f"Results/lorenz_final_params.pkl"
model_path = f"Results/lorenz_final.pt"
params = pickle.load(open(params_path, 'rb'))



# Reconstruct the model
net = full_network(params)
net.load_state_dict(torch.load(model_path, map_location='cpu'))
net.eval()

# Print found coefficient mask
mask = net.coefficient_mask.detach().numpy()
print("SINDy Coefficient Mask (Ξ):")
print(mask)

# Extract SINDy coefficients
Xi = net.sindy_coefficients.detach().numpy()* mask
# Make Xi have only 3 significant digits
Xi = np.round(Xi, 3)
# Print SINDy coefficients
print("SINDy Coefficient Matrix (Ξ):")
print(Xi)

SINDy Coefficient Mask (Ξ):
[[0. 0. 1.]
 [1. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
SINDy Coefficient Matrix (Ξ):
[[  0.          0.         -7.1      ]
 [-10.         -9.6         0.       ]
 [-10.8         0.          0.       ]
 [  0.          0.         -2.6666667]
 [  0.          0.          0.       ]
 [  0.          0.         -3.1      ]
 [  0.         -1.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]
 [  0.          0.          0.       ]

In [15]:
# Reconstruct terms in the library for interpretation
from sindy_utils import sindy_library

latent_dim = params['latent_dim']
poly_order = params['poly_order']
include_sine = params['include_sine']

# Get library term names
def get_term_names(n, poly_order=3, include_sine=False):
    terms = ['1']
    var_names = [f'z{i+1}' for i in range(n)]
    terms.extend(var_names)
    if poly_order >= 2:
        terms.extend([f'{vi}{vj}' for i, vi in enumerate(var_names) for vj in var_names[i:]])
    if poly_order >= 3:
        terms.extend([f'{vi}{vj}{vk}' for i, vi in enumerate(var_names)
                      for j, vj in enumerate(var_names[i:])
                      for vk in var_names[i + j:]])
    if include_sine:
        terms.extend([f'sin({vi})' for vi in var_names])
    return terms

terms = get_term_names(latent_dim, poly_order, include_sine)

# Print dynamics
for i in range(latent_dim):
    eq = f"dz{i+1}/dt = "
    eq += " + ".join([f"{Xi[j,i]:.4f}*{term}" for j, term in enumerate(terms) if abs(Xi[j,i]) > 1e-6])
    print(eq)

dz1/dt = -10.0000*z1 + -10.8000*z2
dz2/dt = -9.6000*z1 + -1.0000*z1z3
dz3/dt = -7.1000*1 + -2.6667*z3 + -3.1000*z1z2


In [20]:
from sindy_utils import library_size
import numpy as np

# Assume Xi_learned has already been loaded
Xi_learned = Xi

# Lorenz ground truth
n_vars = 3
poly_order = params['poly_order']
lib_size = library_size(n_vars, poly_order)

Xi_true = np.zeros((lib_size, 3))

# Polynomial order 1 terms
Xi_true[1, 0] = -10.0     # -sigma * z1
Xi_true[2, 0] = 10.0      # +sigma * z2

Xi_true[1, 1] = -1.0      # -z2
Xi_true[2, 1] = 28.0      # +rho * z1

Xi_true[3, 2] = -8/3      # -beta * z3
Xi_true[4, 2] = 1.0       # +z1*z2
Xi_true[6, 1] = -1.0      # -z1*z3

print("Ground truth coefficient matrix (Ξ):")
print(Xi_true)

# Metrics
nonzero_discovered = np.count_nonzero(Xi_learned)
nonzero_true = np.count_nonzero(Xi_true)
mean_abs_error = np.mean(np.abs(Xi_learned - Xi_true))
exact_sparsity_match = np.all((Xi_learned != 0) == (Xi_true != 0))

print(f"Number of nonzero terms in learned Ξ: {nonzero_discovered}")
print(f"Number of nonzero terms in true Ξ: {nonzero_true}")
print(f"Mean absolute coefficient error: {mean_abs_error:.5f}")
print(f"Exact sparsity pattern match: {exact_sparsity_match}")


Ground truth coefficient matrix (Ξ):
[[  0.           0.           0.        ]
 [-10.          -1.           0.        ]
 [ 10.          28.           0.        ]
 [  0.           0.          -2.66666667]
 [  0.           0.           1.        ]
 [  0.           0.           0.        ]
 [  0.          -1.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]]
Number of nonzero terms in learned Ξ: 7
Number of nonzero terms in true Ξ: 7
Mean absolute coefficient error: 1.14333
Exac

# Analysis of 4000 epochs model

In [39]:
# Load params used for training

params_path = f"Results/lorenz_4000_epochs_params.pkl"
model_path = f"Results/lorenz_4000_epochs.pt"
params = pickle.load(open(params_path, 'rb'))



# Reconstruct the model
net = full_network(params)
net.load_state_dict(torch.load(model_path, map_location='cpu'))
net.eval()

# Print found coefficient mask
mask = net.coefficient_mask.detach().numpy()
print("SINDy Coefficient Mask (Ξ):")
print(mask)

# Extract SINDy coefficients
Xi = net.sindy_coefficients.detach().numpy()* mask
print("SINDy Coefficient Matrix (Ξ):")
print(Xi)

SINDy Coefficient Mask (Ξ):
[[0. 0. 0.]
 [0. 0. 0.]
 [1. 0. 1.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 0. 0.]
 [0. 0. 0.]
 [1. 0. 1.]
 [1. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 1. 0.]
 [0. 1. 1.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]
SINDy Coefficient Matrix (Ξ):
[[ -0.         -0.         -0.       ]
 [  0.          0.         -0.       ]
 [ 20.866858    0.         46.37436  ]
 [  3.601256   -0.          0.       ]
 [ -0.          0.        -10.284109 ]
 [ -0.          0.         36.3537   ]
 [  0.          0.         -0.       ]
 [ -0.          0.          0.       ]
 [ -9.263494   -0.        -12.447245 ]
 [ -2.9673996  -0.          0.       ]
 [ -0.          0.          0.       ]
 [  0.          0.         -0.       ]
 [ -0.         -0.          0.       ]
 [  0.         35.86673     0.       ]
 [  0.         -4.614711   -2.364474 ]
 [ -0.         -0.         -0.       ]
 [ -0.          0.         -0.       ]
 [  0.         -0.          0.       ]

In [40]:
# Reconstruct terms in the library for interpretation
from sindy_utils import sindy_library

latent_dim = params['latent_dim']
poly_order = params['poly_order']
include_sine = params['include_sine']

# Get library term names
def get_term_names(n, poly_order=3, include_sine=False):
    terms = ['1']
    var_names = [f'z{i+1}' for i in range(n)]
    terms.extend(var_names)
    if poly_order >= 2:
        terms.extend([f'{vi}{vj}' for i, vi in enumerate(var_names) for vj in var_names[i:]])
    if poly_order >= 3:
        terms.extend([f'{vi}{vj}{vk}' for i, vi in enumerate(var_names)
                      for j, vj in enumerate(var_names[i:])
                      for vk in var_names[i + j:]])
    if include_sine:
        terms.extend([f'sin({vi})' for vi in var_names])
    return terms

terms = get_term_names(latent_dim, poly_order, include_sine)

# Print dynamics
for i in range(latent_dim):
    eq = f"dz{i+1}/dt = "
    eq += " + ".join([f"{Xi[j,i]:.4f}*{term}" for j, term in enumerate(terms) if abs(Xi[j,i]) > 1e-6])
    print(eq)

dz1/dt = 20.8669*z2 + 3.6013*z3 + -9.2635*z2z3 + -2.9674*z3z3 + 4.5267*z2z3z3 + 0.2374*z3z3z3
dz2/dt = 35.8667*z1z2z2 + -4.6147*z1z2z3
dz3/dt = 46.3744*z2 + -10.2841*z1z1 + 36.3537*z1z2 + -12.4472*z2z3 + -2.3645*z1z2z3


In [41]:
from sindy_utils import library_size
import numpy as np

# Assume Xi_learned has already been loaded
Xi_learned = Xi

# Lorenz ground truth
n_vars = 3
poly_order = params['poly_order']
lib_size = library_size(n_vars, poly_order)

Xi_true = np.zeros((lib_size, 3))

# Polynomial order 1 terms
Xi_true[1, 0] = -10.0     # -sigma * z1
Xi_true[2, 0] = 10.0      # +sigma * z2

Xi_true[1, 1] = -1.0      # -z2
Xi_true[2, 1] = 28.0      # +rho * z1

Xi_true[3, 2] = -8/3      # -beta * z3
Xi_true[4, 2] = 1.0       # +z1*z2
Xi_true[6, 1] = -1.0      # -z1*z3

print("Ground truth coefficient matrix (Ξ):")
print(Xi_true)

# Metrics
nonzero_discovered = np.count_nonzero(Xi_learned)
nonzero_true = np.count_nonzero(Xi_true)
mean_abs_error = np.mean(np.abs(Xi_learned - Xi_true))
exact_sparsity_match = np.all((Xi_learned != 0) == (Xi_true != 0))

print(f"Number of nonzero terms in learned Ξ: {nonzero_discovered}")
print(f"Number of nonzero terms in true Ξ: {nonzero_true}")
print(f"Mean absolute coefficient error: {mean_abs_error:.5f}")
print(f"Exact sparsity pattern match: {exact_sparsity_match}")


Ground truth coefficient matrix (Ξ):
[[  0.           0.           0.        ]
 [-10.          -1.           0.        ]
 [ 10.          28.           0.        ]
 [  0.           0.          -2.66666667]
 [  0.           0.           1.        ]
 [  0.           0.           0.        ]
 [  0.          -1.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]
 [  0.           0.           0.        ]]
Number of nonzero terms in learned Ξ: 13
Number of nonzero terms in true Ξ: 7
Mean absolute coefficient error: 3.72392
Exa